# 컴퓨터 내부에 존재하는 GPU를 이용해서 파이토치 학습시키기
이번 파일의 목적은 컴퓨터가 어떻게 GPU를 사용하는지, CUDA가 무엇인지 등을 학습하고 실제로 PyTorch 프레임워크에서 이를 어떻게 사용하는지에 대해서 알아보는 것 입니다.

# 1. 내가 어떤 GPU를 가지고 있나
> 우선 제 컴퓨터에 어떤 GPU가 있는지 확인해보겠습니다.

`Powershell`에 `nvidia-smi`를 작성해보면 다음과 같은 결과가 나옵니다.

```powershell
PS C:\> nvidia-smi
Fri Aug 14 09:11:50 2026
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 592.00                 Driver Version: 592.00         CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5070 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   40C    P0             20W /  140W |       0MiB /  12227MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+------------------------+----------------------+

+-----------------------------------------------------------------------------------------+
| Processes:                                                                              |
|  GPU   GI   CI              PID   Type   Process name                        GPU Memory |
|        ID   ID                                                               Usage      |
|=========================================================================================|
|  No running processes found                                                             |
+-----------------------------------------------------------------------------------------+
```

이를 통해 현재 컴퓨터에 있는 NVDIA GPU를 확인할 수 있습니다. 출력을 해석해보면 다음과 같이 설명할 수 있습니다.

이는 해당 CUDA Toolkit 버전을 말하는 것이 아닌, 현재 NVIDIA Driver가 지원하는 CUDA 호환 수준을 말하는 것과 같습니다.

여기에서 결과가 정삭적으로 나오면 운영체제와 NVIDIA Driver가 이 GPU를 정상적으로 보고있는지를 확인했다 볼 수 있습니다.

- `NVIDIA-SMI 592.00`: 소프트웨어 드라이버의 버전이 592 버전이라는 뜻입니다.
- `Driver Version: 592.00`
- `CUDA Version: 13.1`: 현재 NVIDIA Driver가 지원하는 최대 CUDA 호환 버전이 13.1 수준이라는 의미입니다.

- `0  NVIDIA GeForce RTX 5070 ...  WDDM`: GPU 번호가 `0`이라는 의미이고 이름이 `NVIDIA GeForce RTX 5070` 이라는 의미입니다. `WDDM`은 `Windows Display Driver Model`을 의미합니다.
- `Bus-Id`: GPU가 컴퓨터 하드웨어 구조상 어디에 붙어있는지를 나타내주는 `PCle 장치 주소`입니다.
- `Disp.A`: `Display Active`의 약자로 현재는 `Off`로 디스플레이 출력에 비활성화되어있다고 볼 수 있습니다.
- `Volatile Uncorr. ECC N/A`: `ECC`는 GPU 메모리 오류를 검출/수정하는 기능과 관계있으며 서버/데이터센터 GPU가 아닌 일반 GeForce 환경에서는 지원하지 않거나 표시 대상이 아니여서 `N/A`가 나올 수 있습니다.
- `Fan Temp Perf Pwr:Usage/Cap`: `N/A  40C  P0   20W / 140W` 형태로 온도, 사용 전력, 성능(P0이 제일 높음) 등을 표현합니다.

In [2]:
# torch와 gpu 확인
import torch

print(torch.__version__)
print(torch.cuda.is_available())

2.13.0+cu130
True


In [3]:
# GPU 0번 칸
print(torch.cuda.get_device_name(0))

NVIDIA GeForce RTX 5070 Ti Laptop GPU


In [4]:
x = torch.tensor([1, 2, 3])

print(x)
print(x.device)

tensor([1, 2, 3])
cpu


In [9]:
x = x.to("cuda")
print(x.device)

cuda:0


# CUDA의 역할
일반적으로 컴퓨터가 연산을 하는 경우에는 CPU가 사용할 수 있는 RAM 공간에 연산할 대상들을 올려 놓은 뒤 해야하는 연산을 CPU가 RAM으로부터 데이터를 가져와 수행하는 방식을 통해 이루어지게 됩니다.

GPU 또한 자신의 RAM 공간을 할당받아 데이터에 접근할 수 있는데 데이터는 기본적으로 CPU의 RAM 공간에 올라가기 때문에 해당 데이터를 GPU 공간으로 보내주며 VRAM의 특정 주소에 결과를 저장하고 등등의 작업을 해야합니다.

해당 작업을 담당하는 계층이 CUDA와 드라이버 계층입니다.

In [11]:
torch.cuda

<module 'torch.cuda' from 'C:\\LANG_CHAIN_2026\\2026-05-19_KDT_lang_chain\\workspace\\.venv\\Lib\\site-packages\\torch\\cuda\\__init__.py'>

In [12]:
x = torch.tensor([1.0, 2.0, 3.0])

print(x.device)

y = x.to("cuda")

print(y.device)

z = y * 10

print(z)
print(z.device)

cpu
cuda:0
tensor([10., 20., 30.], device='cuda:0')
cuda:0


In [13]:
x = torch.tensor([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0]
])

print(x) # tensor
# tensor([[1., 2., 3.],
#         [4., 5., 6.]])

print(type(x)) # tensor.int
# <class 'torch.Tensor'>

print(x.shape) # 2, 3
# torch.Size([2, 3])

print(x.ndim) # 2
# 2

print(x.dtype) # float
# torch.float32

print(x.device) # cpu
# cpu

tensor([[1., 2., 3.],
        [4., 5., 6.]])
<class 'torch.Tensor'>
torch.Size([2, 3])
2
torch.float32
cpu


In [14]:
y = x * 10

print(y) #
# [
#     [10.0, 20.0, 30.0],
#     [40.0, 50.0, 60.0]
# ]

# tensor([[10., 20., 30.],
#         [40., 50., 60.]])

print(y.shape) # torch.Size([2, 3])
# torch.Size([2, 3])

print(y.device) # cpu
# cpu

tensor([[10., 20., 30.],
        [40., 50., 60.]])
torch.Size([2, 3])
cpu


In [16]:
x_gpu = x.to("cuda")

print(x_gpu)
# tensor([[1., 2., 3.],
#         [4., 5., 6.]], device='cuda:0')
print(x_gpu.device) # cuda
# cuda:0

tensor([[1., 2., 3.],
        [4., 5., 6.]], device='cuda:0')
cuda:0


In [19]:
x = torch.tensor([1., 2., 3.])

x_gpu = x.to("cuda")

print("x    :", id(x))
print("x_gpu:", id(x_gpu))

x    : 2860616635632
x_gpu: 2860616635392


In [20]:
y_gpu = x_gpu.to("cuda")

print("x_gpu:", id(x_gpu))
print("y_gpu:", id(y_gpu))

x_gpu: 2860616635392
y_gpu: 2860616635392


In [21]:
x = torch.tensor(2.0, requires_grad=True)

y = x ** 2
y.backward()

print(x.grad)

tensor(4.)


# PyTorch의 핵심인 Autograd
PyTorch는 GPU를 사용하는 기능이 존재하지만 핵심은 Tensor을 연산하며 계산 그래프를 만들어 역전파를 통해 기울기를 구해줄 수 있습니다.

In [22]:
x = torch.tensor(3.0, requires_grad=True)

y = x ** 2

print("x =", x)
print("y = ", y)

x = tensor(3., requires_grad=True)
y =  tensor(9., grad_fn=<PowBackward0>)


In [23]:
print("x.requires_grad =", x.requires_grad)
print("y.requires_grad =", y.requires_grad)

x.requires_grad = True
y.requires_grad = True


In [24]:
print("x.grad =", x.grad)
print("y.grad_fn =", y.grad_fn)
# y.grad_fn = <PowBackward0 object at 0x0000029A09099F30>
# 지수 역전파 객체인 것으로 보임

x.grad = None
y.grad_fn = <PowBackward0 object at 0x0000029A09099F30>


In [25]:
y.backward()
print("x.grad =", x.grad)

x.grad = tensor(6.)


In [26]:
print("y.grad_fn:", y.grad_fn)

y.grad_fn: <PowBackward0 object at 0x0000029A09099F30>


y는 PowBackword0 객첼 아마 수식을 내부적으로 저장하고 있는 것 같다.

어떤 방식으로 저장하며 backward()를 하면 내부적으로 무슨일이 일어나는지,
x의 경우에도 텐서 객체인데 텐서객체의 `**`는 약간 다른 연산을 오버라이딩 해놓았다고 볼 수 있는건가?

In [27]:
print("x.is_leaf =", x.is_leaf)
print("y.is_leaf =", y.is_leaf)

print("x.grad_fn =", x.grad_fn)
print("y.grad_fn =", y.grad_fn)

x.is_leaf = True
y.is_leaf = False
x.grad_fn = None
y.grad_fn = <PowBackward0 object at 0x0000029A09099F30>


In [30]:
x = torch.tensor(2.0, requires_grad=True)

y = 3*x**3 + 2*x**2 + 1
print(y)

tensor(33., grad_fn=<AddBackward0>)


In [31]:
y.backward()

x.grad

tensor(44.)

In [ ]:
x = torch.tensor(2.0, requires_grad=True)

a = x * 3
b = a + 4
y = b ** 2

print(a)
print(b)
print(y)

y.backword()

print(x.grad)